# DAEDALUS Training Setup via GRPO (Hugging Face Edition)

This notebook trains an LLM to design market mechanisms using GRPO via TRL + Unsloth.
It is optimized to be run inside a **Hugging Face Notebook** or **Hugging Face Space (JupyterLab SDK)**.

It will automatically authenticate with your HF account, run the GRPO loop, and push the newly trained designer LLM back to your Hugging Face Models repository (`kabilesh-c/daedalus-designer`).

### Step 0: Install Dependencies
Install the necessary RL logic and environment tools.

In [ ]:
!pip install -q huggingface_hub openenv trl unsloth transformers datasets accelerate
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q fastapi uvicorn pydantic

### Step 1: Hugging Face Authentication & Environment Setup
We use your provided write token so that TRL can automatically push checkpoint stages directly to your HF repository.

In [ ]:
import sys
import os
import json
import random
import numpy as np
from typing import List, Dict, Any

from huggingface_hub import login
# Authenticate with HF
login(token=os.environ.get("HF_TOKEN"))
print("Logged into Hugging Face successfully!")

# Ensure our local 'daedalus' package is accessible
sys.path.insert(0, os.path.abspath('.'))

from daedalus.env import DaedalusEnvironment
from daedalus.models import MechanismConfig, Observation

def _format_prompt(obs: dict) -> str:
    lines = [
        "You are a mechanism designer for a market auction system.",
        "Analyze the current market state and design an optimal mechanism.",
        "",
        f"Round: {obs.get('round_number', 0)} / {obs.get('episode_length', 50)}",
        "",
        "Your goal is to maximize the composite reward R = W × F × P × S"
    ]
    outcomes = obs.get('market_outcomes', [])
    if outcomes:
        lines.append("Recent Market Outcomes:")
        for o in outcomes[-5:]:
            lines.append(
                f"  W={o.get('welfare_ratio', 0):.3f} "
                f"F={1-o.get('gini_coefficient', 0):.3f} "
                f"P={o.get('participation_rate', 1):.3f} "
                f"R={o.get('composite_reward', 0):.3f}"
            )
    lines.extend([
        "",
        "Respond with ONLY a JSON mechanism configuration:",
        '{"auction_type": "first_price|second_price|vcg", "reserve_price": float(0-0.9), '
        '"reveal_reserve": bool, "reveal_competing_bids": bool, "reveal_winner_identity": bool, '
        '"reveal_clearing_price": bool, "reveal_bid_distribution": bool, '
        '"shill_penalty": float(0-3), "withdrawal_penalty": float(0-3), "collusion_penalty": float(0-3), '
        '"coalition_policy": "allow|restrict|penalize_suspected|penalize_confirmed"}',
    ])
    return "\n".join(lines)

def _random_mechanism() -> dict:
    return {
        "auction_type": random.choice(["first_price", "second_price", "vcg"]),
        "reserve_price": random.uniform(0.05, 0.5),
        "reveal_reserve": random.random() > 0.5,
        "reveal_competing_bids": random.random() > 0.7,
        "reveal_winner_identity": random.random() > 0.5,
        "reveal_clearing_price": random.random() > 0.3,
        "reveal_bid_distribution": random.random() > 0.7,
        "shill_penalty": random.uniform(0, 2),
        "withdrawal_penalty": random.uniform(0, 1),
        "collusion_penalty": random.uniform(0, 2),
        "coalition_policy": random.choice(["allow", "restrict", "penalize_suspected"]),
    }

### Step 2: Build the Training Dataset
We extract states directly from your DAEDALUS environment and map them into LLM prompts.

In [ ]:
def generate_training_prompts(n_prompts: int = 500) -> List[Dict[str, str]]:
    print(f"\nGenerating {n_prompts} training prompts...")
    prompts = []
    env = DaedalusEnvironment()
    for i in range(n_prompts):
        obs_dict = env.reset()
        prompt_text = _format_prompt(obs_dict)
        prompts.append({"prompt": prompt_text})

        # Also generate mid-episode prompts
        for step in range(random.randint(1, 5)):
            obs_dict, _, done, _ = env.step(_random_mechanism())
            if done: break
            prompts.append({"prompt": _format_prompt(obs_dict)})

    print(f"Generated {len(prompts)} prompts.")
    return prompts

from datasets import Dataset
dataset = Dataset.from_list(generate_training_prompts(n_prompts=200))
print("Dataset created! Ready for GRPO.")

### Step 3: Define the GRPO Evaluator Loop
Takes the mechanism JSON outputted by the LLM and plays it inside your simulator to get the reward.

In [ ]:
eval_env = DaedalusEnvironment()

def evaluate_mechanism(generated_text: str, env: DaedalusEnvironment) -> float:
    try:
        j_start = generated_text.find('{')
        j_end = generated_text.rfind('}') + 1
        if j_start >= 0 and j_end > j_start:
            mech = json.loads(generated_text[j_start:j_end])
        else:
            return -0.5
    except:
        return -0.5
    _, reward, _, _ = env.step(mech)
    return reward

def grpo_reward_fn(completions: List[str], prompts: List[str] = None, **kwargs) -> List[float]:
    rewards = []
    for comp in completions:
        eval_env.reset()
        rewards.append(evaluate_mechanism(comp, eval_env))
    return rewards

### Step 4: Hugging Face Native GRPO Setup via Unsloth & TRL
Runs the trainer and continuously syncs output back to your Hub profile (`kabilesh-c/daedalus-designer`).

In [ ]:
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer

# 1. Load Unsloth Model
model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=4096,
    load_in_4bit=True,
)

# 2. Apply LoRA
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth",
)

# 3. GRPO Config for HF Sync
training_args = GRPOConfig(
    output_dir="./daedalus-checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_generations=8,
    max_completion_length=512,
    learning_rate=5e-6,
    logging_steps=10,
    save_steps=200,
    bf16=True,
    push_to_hub=True,
    hub_model_id="kabilesh-c/daedalus-designer",
    hub_strategy="every_save",
    report_to="none"
)

# 4. Training
trainer = GRPOTrainer(
    model=model,
    args=training_args,\n    tokenizer=tokenizer,
    reward_funcs=[grpo_reward_fn],
    train_dataset=dataset,
)

print("Starting Training Pipeline...")
trainer.train()

### Step 5: Save & Export to Hugging Face
Push the final merged adapters to your model repository.

In [ ]:
print("\n\ud83d\ude80 Pushing final adapter to Hugging Face...")
model.push_to_hub("kabilesh-c/daedalus-designer")
tokenizer.push_to_hub("kabilesh-c/daedalus-designer")
print("Model successfully deployed to HF Hub!")